# LeetCode 3382 — Maximum Area Rectangle With Point Constraints II bằng Python

- Link: https://leetcode.com/problems/maximum-area-rectangle-with-point-constraints-ii/
- Độ khó: **Hard**
- Chủ đề: **Quét theo trục x từ phải sang trái**, **cây phân đoạn min** trên trục y (nén tọa độ) để tìm "cột gần nhất bên phải chạm khoảng `[y1, y2]`", kiểm tra cặp điểm **kề nhau** trên cột

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxRectangleArea(self, xCoord: List[int], yCoord: List[int]) -> int:
```

## 1. Mục tiêu học

1. Đặc trưng hóa hình chữ nhật **rỗng** (không điểm nào bên trong hay trên cạnh, ngoài 4 góc): cạnh trái gồm hai điểm **kề nhau** theo `y` trên cột `x1`, cạnh phải là cột `x2` **gần nhất** bên phải có điểm trong `[y1, y2]`, và trên cột đó `y1`, `y2` cũng **kề nhau**.
2. Quét các cột từ phải sang trái, duy trì với mỗi `y` "cột gần nhất bên phải có điểm ở độ cao `y`".
3. Trả lời truy vấn "min trên đoạn `[y1, y2]`" bằng **cây phân đoạn** (nén `y`).
4. Đếm số ứng viên: chỉ các cặp điểm kề nhau trên cùng cột — tổng `O(n)` ứng viên.
5. Kiểm chứng bằng vét cạn và bản `O(n²)`.

## 2. Đề bài & ví dụ

Cho `n` điểm phân biệt. Tìm diện tích lớn nhất của hình chữ nhật có cạnh song song trục, **4 góc là 4 điểm đã cho**, và **không có điểm nào khác** nằm bên trong hoặc trên cạnh. Không có ⇒ `-1`.

**Ví dụ 1:** `(1,1),(1,3),(3,1),(3,3)` → `4`. **Ví dụ 2:** thêm `(2,2)` → `-1`. **Ví dụ 3:** `(1,1),(1,3),(3,1),(3,3),(1,2),(3,2)` → `2`.

**Ràng buộc:** `n ≤ 2·10^5`, tọa độ ≤ `8·10^7`.

In [1]:
from typing import List
from collections import defaultdict
import bisect

examples = [([1, 1, 3, 3], [1, 3, 1, 3], 4), ([1, 1, 3, 3, 2], [1, 3, 1, 3, 2], -1),
            ([1, 1, 3, 3, 1, 3], [1, 3, 1, 3, 2, 2], 2)]
for X, Y, exp in examples:
    print(list(zip(X, Y)), '-> kỳ vọng', exp)

[(1, 1), (1, 3), (3, 1), (3, 3)] -> kỳ vọng 4
[(1, 1), (1, 3), (3, 1), (3, 3), (2, 2)] -> kỳ vọng -1
[(1, 1), (1, 3), (3, 1), (3, 3), (1, 2), (3, 2)] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Kiểm tra một hình chữ nhật có rỗng

In [2]:
def is_empty_rect(points, x1, y1, x2, y2):
    corners = {(x1, y1), (x1, y2), (x2, y1), (x2, y2)}
    if not corners <= points:
        return False
    return all(not (x1 <= x <= x2 and y1 <= y <= y2) or (x, y) in corners for x, y in points)

P = set(zip(*examples[2][:2]))
print(is_empty_rect(P, 1, 1, 3, 3), is_empty_rect(P, 1, 2, 3, 3), is_empty_rect(P, 1, 1, 3, 2))

False True True


### 3.2. Nén tọa độ và cây phân đoạn min (không đệ quy)

`y` tới `8·10^7` ⇒ nén về chỉ số `0..m−1`. Cây phân đoạn dạng mảng `2·size`: cập nhật điểm và truy vấn min đoạn `[l, r]` bằng hai con trỏ đi lên.

In [3]:
class MinTree:
    def __init__(self, m):
        self.size = 1
        while self.size < m:
            self.size *= 2
        self.t = [float('inf')] * (2 * self.size)

    def update(self, i, v):
        i += self.size
        self.t[i] = v
        i //= 2
        while i:
            self.t[i] = min(self.t[2 * i], self.t[2 * i + 1])
            i //= 2

    def query(self, l, r):                  # min trên [l, r]
        res = float('inf')
        l += self.size; r += self.size + 1
        while l < r:
            if l & 1:
                res = min(res, self.t[l]); l += 1
            if r & 1:
                r -= 1; res = min(res, self.t[r])
            l //= 2; r //= 2
        return res

mt = MinTree(5)
for i, v in enumerate([7, 3, 9, 4, 8]):
    mt.update(i, v)
print(mt.query(0, 4), mt.query(2, 4), mt.query(2, 2))

3 4 9


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Xét hình chữ nhật rỗng `[x1, x2] × [y1, y2]`:

- Trên cạnh trái (cột `x1`) không có điểm nào giữa `y1` và `y2` ⇒ `y1`, `y2` là hai điểm **liên tiếp** trên cột `x1`.
- Không có điểm nào có `x ∈ (x1, x2)` và `y ∈ [y1, y2]` ⇒ `x2` là cột **gần nhất** bên phải có điểm trong `[y1, y2]`.
- Trên cạnh phải, `(x2, y1)` và `(x2, y2)` phải tồn tại và không có điểm giữa ⇒ `y1`, `y2` liên tiếp trên cột `x2`.

Ví dụ 3: cột `x = 1` có `y = 1, 2, 3` ⇒ cặp kề `(1, 2)` và `(2, 3)`. Với `(2, 3)`: cột gần nhất bên phải chạm `[2, 3]` là `x = 3`, có `y = 1, 2, 3` ⇒ `(2, 3)` kề ⇒ diện tích `2 · 1 = 2`.

## 5. Cách 1 — Brute Force: mọi cặp đường chéo

In [4]:
def max_area_brute(X, Y):
    P = set(zip(X, Y)); pts = list(P); best = -1
    for x1, y1 in pts:
        for x2, y2 in pts:
            if x2 > x1 and y2 > y1 and is_empty_rect(P, x1, y1, x2, y2):
                best = max(best, (x2 - x1) * (y2 - y1))
    return best

for X, Y, exp in examples:
    print(max_area_brute(X, Y), '| kỳ vọng', exp)

4 | kỳ vọng 4
-1 | kỳ vọng -1
2 | kỳ vọng 2


**Độ phức tạp:** `O(n³)` — chỉ cho `n ≤ 15`.

## 6. Ý tưởng thuật toán / bước đột phá

Gom điểm theo cột; quét các cột theo `x` **giảm dần**. Cây phân đoạn lưu tại vị trí `y` (đã nén) giá trị `x` **nhỏ nhất** của các cột đã quét (tức cột gần nhất bên phải) có điểm ở độ cao `y`.

Tại cột `x` (đã sắp `y`): với mỗi cặp kề `(y1, y2)`:
1. `x2 = query(y1, y2)` — cột gần nhất bên phải chạm `[y1, y2]` (truy vấn **trước** khi chèn cột hiện tại).
2. Nếu `x2` tồn tại và `(y1, y2)` là cặp kề trên cột `x2` ⇒ ứng viên `(x2 − x)(y2 − y1)`.

Sau đó cập nhật mọi `y` của cột `x` với giá trị `x`. Tổng `O(n log n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Duyệt cột bên phải tuyến tính (`O(n · số cột)`)

In [5]:
def max_area_quadratic(X, Y):
    cols = defaultdict(list)
    for x, y in zip(X, Y):
        cols[x].append(y)
    xs = sorted(cols)
    for x in xs:
        cols[x].sort()
    best = -1
    for ci, x in enumerate(xs):
        ys = cols[x]
        for y1, y2 in zip(ys, ys[1:]):
            for x2 in xs[ci + 1:]:
                c = cols[x2]
                k = bisect.bisect_left(c, y1)
                if k < len(c) and c[k] <= y2:              # cột đầu tiên chạm [y1, y2]
                    if c[k] == y1 and k + 1 < len(c) and c[k + 1] == y2:
                        best = max(best, (x2 - x) * (y2 - y1))
                    break
    return best

for X, Y, exp in examples:
    print(max_area_quadratic(X, Y), '| kỳ vọng', exp)

4 | kỳ vọng 4
-1 | kỳ vọng -1
2 | kỳ vọng 2


### 7.2. Cách 3 — Quét phải → trái + cây phân đoạn min (lời giải chính)

In [6]:
def max_area_segtree(X, Y):
    cols = defaultdict(list)
    for x, y in zip(X, Y):
        cols[x].append(y)
    ys = sorted(set(Y))
    yi = {v: i for i, v in enumerate(ys)}
    tree = MinTree(len(ys))
    pairs = {}
    best = -1
    for x in sorted(cols, reverse=True):
        c = sorted(cols[x])
        pairs[x] = set(zip(c, c[1:]))
        for y1, y2 in zip(c, c[1:]):
            x2 = tree.query(yi[y1], yi[y2])
            if x2 != float('inf') and (y1, y2) in pairs[x2]:
                best = max(best, (x2 - x) * (y2 - y1))
        for y in c:
            tree.update(yi[y], x)
    return best

for X, Y, exp in examples:
    print(max_area_segtree(X, Y), '| kỳ vọng', exp)

4 | kỳ vọng 4
-1 | kỳ vọng -1
2 | kỳ vọng 2


### 7.3. Cách 4 — Đối xứng: quét theo `y` (hoán đổi vai trò trục)

Hình chữ nhật rỗng cũng đặc trưng được bằng hai điểm kề nhau trên cùng **hàng** và hàng gần nhất phía trên. Hoán đổi `X ↔ Y` và gọi lại cách 3 — kết quả phải giống nhau (một kiểm tra tốt cho cài đặt).

In [7]:
def max_area_swapped(X, Y):
    return max_area_segtree(Y, X)

for X, Y, exp in examples:
    print(max_area_swapped(X, Y), '| kỳ vọng', exp)

4 | kỳ vọng 4
-1 | kỳ vọng -1
2 | kỳ vọng 2


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 3.

In [8]:
def trace(X, Y):
    cols = defaultdict(list)
    for x, y in zip(X, Y):
        cols[x].append(y)
    ys = sorted(set(Y)); yi = {v: i for i, v in enumerate(ys)}
    tree = MinTree(len(ys)); pairs = {}; best = -1
    for x in sorted(cols, reverse=True):
        c = sorted(cols[x]); pairs[x] = set(zip(c, c[1:]))
        for y1, y2 in zip(c, c[1:]):
            x2 = tree.query(yi[y1], yi[y2])
            ok = x2 != float('inf') and (y1, y2) in pairs[x2]
            msg = f"cột gần nhất chạm [{y1},{y2}] là x={x2 if x2 != float('inf') else '∞'}"
            if ok:
                best = max(best, (x2 - x) * (y2 - y1))
                msg += f", ({y1},{y2}) kề trên cột đó -> diện tích {(x2 - x) * (y2 - y1)}"
            print(f"cột x={x}, cặp ({y1},{y2}): {msg}")
        for y in c:
            tree.update(yi[y], x)
    print('đáp án:', best)

trace(*examples[2][:2])

cột x=3, cặp (1,2): cột gần nhất chạm [1,2] là x=∞
cột x=3, cặp (2,3): cột gần nhất chạm [2,3] là x=∞
cột x=1, cặp (1,2): cột gần nhất chạm [1,2] là x=3, (1,2) kề trên cột đó -> diện tích 2
cột x=1, cặp (2,3): cột gần nhất chạm [2,3] là x=3, (2,3) kề trên cột đó -> diện tích 2
đáp án: 2


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def maxRectangleArea(self, xCoord: List[int], yCoord: List[int]) -> int:
        cols = {}
        for x, y in zip(xCoord, yCoord):
            cols.setdefault(x, []).append(y)
        ys = sorted(set(yCoord))
        yi = {v: i for i, v in enumerate(ys)}
        size = 1
        while size < len(ys):
            size *= 2
        INF = float('inf')
        t = [INF] * (2 * size)
        pairs = {}
        best = -1
        for x in sorted(cols, reverse=True):
            c = sorted(cols[x])
            pairs[x] = set(zip(c, c[1:]))
            for y1, y2 in zip(c, c[1:]):
                l, r = yi[y1] + size, yi[y2] + size + 1
                res = INF
                while l < r:
                    if l & 1:
                        if t[l] < res:
                            res = t[l]
                        l += 1
                    if r & 1:
                        r -= 1
                        if t[r] < res:
                            res = t[r]
                    l >>= 1
                    r >>= 1
                if res != INF and (y1, y2) in pairs[res]:
                    area = (res - x) * (y2 - y1)
                    if area > best:
                        best = area
            for y in c:
                i = yi[y] + size
                t[i] = x
                i >>= 1
                while i:
                    v = t[2 * i] if t[2 * i] < t[2 * i + 1] else t[2 * i + 1]
                    if t[i] == v:
                        break
                    t[i] = v
                    i >>= 1
        return best

for X, Y, exp in examples:
    got = Solution().maxRectangleArea(X, Y)
    print(got, 'OK' if got == exp else 'SAI')

4 OK
-1 OK
2 OK


Mẹo hiệu năng trong bản nộp: khi cập nhật giá trị **giảm** dần (`x` giảm), có thể dừng đi lên sớm khi nút cha không đổi.

## 10. Vì sao thuật toán đúng

1. **Điều kiện cần (mục 4):** mọi hình chữ nhật rỗng có cạnh trái là một cặp kề trên cột `x1`, cạnh phải là cột gần nhất bên phải chạm `[y1, y2]`, và `(y1, y2)` kề trên cột đó.
2. **Điều kiện đủ:** nếu ba điều kiện thỏa thì: không điểm nào trên cạnh trái (kề), không điểm nào có `x ∈ (x1, x2)` trong `[y1, y2]` (gần nhất), không điểm nào trên cạnh phải ngoài hai góc (kề), và cạnh trên/dưới (`y = y1` hoặc `y2`, `x` giữa) cũng bị loại bởi điều kiện "gần nhất" ⇒ rỗng.
3. **Cây phân đoạn:** khi xử lý cột `x`, cây chứa đúng các cột `> x`; giá trị ở `y` là cột nhỏ nhất (gần nhất) có điểm ở `y` ⇒ `query(y1, y2)` là cột gần nhất chạm đoạn.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp chéo | `O(n³)` | `O(n)` |
| 2. Duyệt cột tuyến tính | `O(n · C · log)` (`C` số cột) | `O(n)` |
| 3. Cây phân đoạn | `O(n log n)` | `O(n)` |
| 4. Hoán đổi trục | `O(n log n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'quadratic': max_area_quadratic, 'segtree': max_area_segtree, 'swapped': max_area_swapped,
         'submit': lambda X, Y: Solution().maxRectangleArea(X, Y)}
edge = [
    ([0], [0], -1),
    ([0, 0, 5, 5], [0, 5, 0, 5], 25),
    ([0, 0, 5, 5, 2], [0, 5, 0, 5, 5], -1),          # điểm trên cạnh trên
    ([0, 0, 5, 5, 9, 9], [0, 5, 0, 5, 0, 5], 25),     # chọn hình lớn nhất trong hai hình rỗng
    ([0, 0, 2, 2, 1], [0, 2, 0, 2, 7], 4),            # điểm ngoài hình không ảnh hưởng
]
for X, Y, exp in examples + edge:
    assert max_area_brute(X, Y) == exp, (X, Y, max_area_brute(X, Y))
    for name, f in impls.items():
        assert f(X, Y) == exp, (name, X, Y)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3382)
cnt = 0
for _ in range(1500):
    n = rng.randint(1, 13); P = set()
    g = rng.randint(2, 5)
    while len(P) < min(n, (g + 1) ** 2):
        P.add((rng.randint(0, g), rng.randint(0, g)))
    X, Y = [p[0] for p in P], [p[1] for p in P]
    exp = max_area_brute(X, Y)
    for name, f in impls.items():
        assert f(X, Y) == exp, (name, P)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ điểm ngẫu nhiên.')

Khớp vét cạn trên 1500 bộ điểm ngẫu nhiên.


### 12.3. Ca lớn `n = 2·10^5`

In [12]:
import time
def rand_points(n, R):
    P = set()
    while len(P) < n:
        P.add((rng.randint(0, R), rng.randint(0, R)))
    return [p[0] for p in P], [p[1] for p in P]

cases = {
    'lưới dày 2000x2000': rand_points(2 * 10 ** 5, 2000),
    'thưa trong 8e7': rand_points(2 * 10 ** 5, 8 * 10 ** 7),
    'lưới đầy 447x447': ([i for i in range(447) for _ in range(447)], [j for _ in range(447) for j in range(447)]),
}
for name, (X, Y) in cases.items():
    t0 = time.time(); a = Solution().maxRectangleArea(X, Y); t1 = time.time() - t0
    print(f"{name:20s}: {a:8d} | {t1:.2f}s")

lưới dày 2000x2000  :      171 | 0.44s


thưa trong 8e7      :       -1 | 1.67s


lưới đầy 447x447    :        1 | 0.24s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra 4 góc tồn tại

In [13]:
def wrong_corners_only(X, Y):
    P = set(zip(X, Y)); best = -1
    for x1, y1 in P:
        for x2, y2 in P:
            if x2 > x1 and y2 > y1 and (x1, y2) in P and (x2, y1) in P:
                best = max(best, (x2 - x1) * (y2 - y1))
    return best

X, Y, _ = examples[1]
print('sai :', wrong_corners_only(X, Y), '| đúng:', Solution().maxRectangleArea(X, Y))

sai : 4 | đúng: -1


**Vì sao sai:** điểm `(2, 2)` nằm **bên trong** hình chữ nhật.

### Lỗi 2 — Chỉ kiểm tra bên trong (quên các điểm trên cạnh)

In [14]:
def wrong_interior_only(X, Y):
    P = set(zip(X, Y)); best = -1
    for x1, y1 in P:
        for x2, y2 in P:
            if x2 > x1 and y2 > y1 and (x1, y2) in P and (x2, y1) in P:
                if all(not (x1 < x < x2 and y1 < y < y2) for x, y in P):
                    best = max(best, (x2 - x1) * (y2 - y1))
    return best

X, Y = [0, 0, 5, 5, 2], [0, 5, 0, 5, 5]
print('sai :', wrong_interior_only(X, Y), '| đúng:', Solution().maxRectangleArea(X, Y))

sai : 25 | đúng: -1


**Vì sao sai:** đề cấm cả điểm **trên cạnh**; `(2, 5)` nằm trên cạnh trên.

### Lỗi 3 — Không kiểm tra `(y1, y2)` kề nhau trên cột bên phải

In [15]:
def wrong_no_right_adjacency(X, Y):
    cols = defaultdict(set)
    for x, y in zip(X, Y):
        cols[x].add(y)
    ys = sorted(set(Y)); yi = {v: i for i, v in enumerate(ys)}
    tree = MinTree(len(ys)); best = -1
    for x in sorted(cols, reverse=True):
        c = sorted(cols[x])
        for y1, y2 in zip(c, c[1:]):
            x2 = tree.query(yi[y1], yi[y2])
            if x2 != float('inf') and y1 in cols[x2] and y2 in cols[x2]:     # SAI: thiếu "kề nhau"
                best = max(best, (x2 - x) * (y2 - y1))
        for y in c:
            tree.update(yi[y], x)
    return best

X, Y = [0, 0, 4, 4, 4], [0, 4, 0, 4, 2]
print('sai :', wrong_no_right_adjacency(X, Y), '| đúng:', Solution().maxRectangleArea(X, Y))

sai : 16 | đúng: -1


**Vì sao sai:** `(4, 2)` nằm trên cạnh phải của hình `[0,4] × [0,4]`.

### Lỗi 4 — Tìm cột gần nhất chỉ theo điểm ở **đúng** `y1` hoặc `y2`

In [16]:
def wrong_only_endpoints(X, Y):
    cols = defaultdict(list)
    for x, y in zip(X, Y):
        cols[x].append(y)
    nearest = {}                                   # y -> cột gần nhất bên phải có điểm ở y
    pairs = {}; best = -1
    for x in sorted(cols, reverse=True):
        c = sorted(cols[x]); pairs[x] = set(zip(c, c[1:]))
        for y1, y2 in zip(c, c[1:]):
            x2 = min(nearest.get(y1, float('inf')), nearest.get(y2, float('inf')))   # SAI
            if x2 != float('inf') and (y1, y2) in pairs[x2]:
                best = max(best, (x2 - x) * (y2 - y1))
        for y in c:
            nearest[y] = x
    return best

X, Y, _ = examples[1]
print('sai :', wrong_only_endpoints(X, Y), '| đúng:', Solution().maxRectangleArea(X, Y))

sai : 4 | đúng: -1


**Vì sao sai:** điểm chặn có thể ở độ cao **giữa** `y1` và `y2` (như `(2, 2)`), nên phải truy vấn min trên **cả đoạn** `[y1, y2]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hình chữ nhật rỗng với góc là điểm cho trước" ⇒ góc kề nhau trên cột + cột gần nhất.
- "Điểm gần nhất theo một trục trong một đoạn của trục kia" ⇒ quét + cây phân đoạn min/max.
- Nén tọa độ khi giá trị lớn.
- Kiểm tra tính đối xứng (hoán đổi trục) là cách tự kiểm tra cài đặt rẻ.

## 15. Biến thể của bài

1. **LC 3380 — bản I** (`n ≤ 10`): vét cạn là đủ.
2. Đếm số hình chữ nhật rỗng ⇒ cùng duyệt, đếm thay vì lấy max.
3. Cho phép điểm trên cạnh (chỉ cấm bên trong) ⇒ điều kiện "kề" và "gần nhất" thay bằng mở/đóng khác.
4. Bài liên quan: LC 939 (Minimum Area Rectangle) — không có ràng buộc rỗng.

In [17]:
# Biến thể 2: đếm số hình chữ nhật rỗng
def count_empty_rects(X, Y):
    cols = defaultdict(list)
    for x, y in zip(X, Y):
        cols[x].append(y)
    ys = sorted(set(Y)); yi = {v: i for i, v in enumerate(ys)}
    tree = MinTree(len(ys)); pairs = {}; cnt = 0
    for x in sorted(cols, reverse=True):
        c = sorted(cols[x]); pairs[x] = set(zip(c, c[1:]))
        for y1, y2 in zip(c, c[1:]):
            x2 = tree.query(yi[y1], yi[y2])
            if x2 != float('inf') and (y1, y2) in pairs[x2]:
                cnt += 1
        for y in c:
            tree.update(yi[y], x)
    return cnt

def count_brute(X, Y):
    P = set(zip(X, Y))
    return sum(1 for x1, y1 in P for x2, y2 in P if x2 > x1 and y2 > y1 and is_empty_rect(P, x1, y1, x2, y2))

rng = random.Random(15)
for _ in range(400):
    P = set()
    while len(P) < rng.randint(1, 12):
        P.add((rng.randint(0, 4), rng.randint(0, 4)))
    X, Y = [p[0] for p in P], [p[1] for p in P]
    assert count_empty_rects(X, Y) == count_brute(X, Y)
print('Biến thể đếm khớp vét cạn; lưới 3x3 đầy có', count_empty_rects([i for i in range(3) for _ in range(3)], [j for _ in range(3) for j in range(3)]), 'hình rỗng.')

Biến thể đếm khớp vét cạn; lưới 3x3 đầy có 4 hình rỗng.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi cặp chéo | `O(n³)` | Oracle |
| 2 | Duyệt cột tuyến tính | `O(n · C)` | Bản I/II nhỏ |
| 3 | Quét + cây phân đoạn | `O(n log n)` | **Nộp bài** |
| 4 | Hoán đổi trục | `O(n log n)` | Tự kiểm tra |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `(0,0), (0,1), (1,0), (1,1), (2,0), (2,1)`.

**Bài 2 (tính tay):** thêm điểm `(1, 5)` vào bài 1.

**Bài 3 (điền TODO):** trả về **4 góc** của một hình chữ nhật đạt diện tích lớn nhất.

**Bài 4 (tự viết lại):** biến thể 3 — cho phép điểm trên cạnh, chỉ cấm điểm **bên trong**.

**Bài 5 (mini challenge):** LC 939 — diện tích **nhỏ nhất** của hình chữ nhật (không cần rỗng).

In [18]:
def best_corners_todo(X, Y):
    # TODO
    raise NotImplementedError

def max_area_interior_only_todo(X, Y):
    # TODO
    raise NotImplementedError

def min_area_rect_todo(points):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Hai hình `1 × 1` rỗng. (2) Hình `2 × 1` chứa `(1, 0)`, `(1, 1)` trên cạnh. (3) `1`.

**Bài 2.** (1) `(1, 5)` nằm ngoài các hình. (2) Không ảnh hưởng. (3) `1`.

**Bài 3.** (1) Lưu `(x, y1, x2, y2)` khi cập nhật. (2) Trả 4 góc. (3) Kiểm tra bằng `is_empty_rect`.

**Bài 4.** (1) Vét cạn với điều kiện bên trong nghiêm ngặt. (2) Phiên bản nhanh cần cấu trúc khác. (3) So sánh với bản gốc.

**Bài 5.** (1) Gom điểm theo cặp `y` trên mỗi cột. (2) Lưu cột cuối cùng thấy mỗi cặp. (3) Diện tích nhỏ nhất.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
X1, Y1 = [0, 0, 1, 1, 2, 2], [0, 1, 0, 1, 0, 1]
assert Solution().maxRectangleArea(X1, Y1) == 1 == max_area_brute(X1, Y1)
assert Solution().maxRectangleArea(X1 + [1], Y1 + [5]) == 1

def best_corners(X, Y):
    cols = defaultdict(list)
    for x, y in zip(X, Y):
        cols[x].append(y)
    ys = sorted(set(Y)); yi = {v: i for i, v in enumerate(ys)}
    tree = MinTree(len(ys)); pairs = {}; best = (-1, None)
    for x in sorted(cols, reverse=True):
        c = sorted(cols[x]); pairs[x] = set(zip(c, c[1:]))
        for y1, y2 in zip(c, c[1:]):
            x2 = tree.query(yi[y1], yi[y2])
            if x2 != float('inf') and (y1, y2) in pairs[x2] and (x2 - x) * (y2 - y1) > best[0]:
                best = ((x2 - x) * (y2 - y1), (x, y1, x2, y2))
        for y in c:
            tree.update(yi[y], x)
    return best

area, rect = best_corners(*examples[2][:2])
assert is_empty_rect(set(zip(*examples[2][:2])), *rect)
print('Bài 1: 1 | Bài 2: 1 | Bài 3:', area, rect)

Bài 1: 1 | Bài 2: 1 | Bài 3: 2 (1, 1, 3, 2)


In [20]:
# Bài 4, 5
def max_area_interior_only(X, Y):
    P = set(zip(X, Y)); best = -1
    for x1, y1 in P:
        for x2, y2 in P:
            if x2 > x1 and y2 > y1 and (x1, y2) in P and (x2, y1) in P:
                if all(not (x1 < x < x2 and y1 < y < y2) for x, y in P):
                    best = max(best, (x2 - x1) * (y2 - y1))
    return best

def min_area_rect(points):
    cols = defaultdict(list)
    for x, y in points:
        cols[x].append(y)
    last, best = {}, float('inf')
    for x in sorted(cols):
        c = sorted(cols[x])
        for i in range(len(c)):
            for j in range(i + 1, len(c)):
                key = (c[i], c[j])
                if key in last:
                    best = min(best, (x - last[key]) * (c[j] - c[i]))
                last[key] = x
    return 0 if best == float('inf') else best

assert max_area_interior_only([0, 0, 5, 5, 2], [0, 5, 0, 5, 5]) == 25
assert min_area_rect([[1, 1], [1, 3], [3, 1], [3, 3], [2, 2]]) == 4
assert min_area_rect([[1, 1], [1, 3], [3, 1], [3, 3], [4, 1], [4, 3]]) == 2          # ví dụ LC 939
print('Bài 4, 5 đúng.')

Bài 4, 5 đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
gom theo cột; nén y; cây min trên y (giá trị = cột gần nhất bên phải)
for x giảm dần:
    c = sorted(y của cột x); pairs[x] = cặp kề
    for (y1, y2) kề: x2 = query(y1, y2); if (y1, y2) in pairs[x2]: ans = max(ans, (x2-x)(y2-y1))
    for y in c: update(y, x)
```

**Checklist:**

- [ ] Cấm điểm trên **cạnh** lẫn bên trong?
- [ ] Cạnh trái: chỉ cặp `y` **kề nhau**?
- [ ] Truy vấn trên **cả đoạn** `[y1, y2]`, **trước** khi chèn cột hiện tại?
- [ ] Kiểm tra `(y1, y2)` kề trên cột `x2`?
- [ ] Không có hình ⇒ `-1`?